# Module 05 · Unit 01
# Graph Traversal and Reachability

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Attack Graphs & Reachability \[AG\] |
| **Refresher** | Module 04 · Unit 01–02 — Graph Definitions, Directed Graphs |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain the difference between **Breadth-First Search (BFS)** and **Depth-First Search (DFS)**
2. Trace BFS and DFS step-by-step on a graph, identifying the frontier and visited sets
3. Prove the correctness of BFS for shortest-hop path finding
4. Compute the **reachability set** of any vertex using BFS
5. Use DFS to detect cycles and classify back edges
6. Apply traversal to formal security claims: prove or disprove that lateral movement is possible


## 🔣 Symbol Primer

Module 05 introduces algorithmic notation alongside the graph notation from Module 04.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $R(v)$ | Reachability set | "reachable from $v$" | All vertices reachable from $v$ via directed paths |
| $d(s, v)$ | Hop distance | "distance from $s$ to $v$" | Minimum number of edges in any path $s \rightsquigarrow v$ |
| $Q$ | Queue | "queue" | FIFO data structure — first in, first out |
| $S$ | Stack | "stack" | LIFO data structure — last in, first out |
| $\text{BFS}(G, s)$ | Breadth-First Search | "BFS from $s$" | Explores vertices layer by layer, nearest first |
| $\text{DFS}(G, s)$ | Depth-First Search | "DFS from $s$" | Explores as deep as possible before backtracking |
| $\pi(v)$ | Parent / predecessor | "parent of $v$" | The vertex from which $v$ was first discovered |
| $\delta(s, v)$ | True shortest distance | "shortest path distance" | The minimum hop count; BFS computes this exactly |

> **The connection to Module 02.** Reachability is a $\exists$ claim:
> $u \rightsquigarrow v \equiv \exists$ a directed path from $u$ to $v$.
> A traversal algorithm proves or disproves this claim constructively —
> it either finds the path (witness) or exhausts the search space (proof of absence).
> BFS and DFS are the computational implementations of the Module 02 audit framework,
> now running on graph structure instead of a user database.

---


## 1 · The Reachability Question

In Module 04 we established that the core security question on a graph is:

$$\text{compromised}(s) \wedge s \rightsquigarrow t \;\Rightarrow\; \text{at risk}(t)$$

*"If source $s$ is compromised and target $t$ is reachable from $s$, then $t$ is at risk."*

This is a conditional — and from Module 01 we know the violation condition is
$\text{compromised}(s) \wedge s \rightsquigarrow t \wedge \neg\,\text{detected}$ — the
attacker moves from $s$ to $t$ undetected. The formal security question reduces to:

$$\text{Is } t \in R(s)?$$

where $R(s) = \{v \in V \mid s \rightsquigarrow v\}$ is the **reachability set** of $s$.

Computing $R(s)$ for every possible entry point $s$ is the foundation of attack
graph analysis. Two algorithms accomplish this with different tradeoffs: BFS and DFS.

### The Lateral Movement Problem

**Lateral movement** is the process by which an attacker, having gained an initial
foothold on one system, progressively moves to other systems by exploiting trust
relationships, credentials, and network connections.

Formally: starting from a compromised vertex $s$ (the initial access point), find
all vertices $t$ such that $s \rightsquigarrow t$ — the full reachability set is the
**blast radius** of the initial compromise.

The attacker wants the *shortest* path to the target. The defender wants to ensure
the *minimum* path length is maximally large — ideally infinite (no path exists).


## 2 · Breadth-First Search (BFS)

**BFS** explores a graph layer by layer — first all vertices at distance 1, then
all at distance 2, and so on. It uses a **queue** $Q$ (FIFO — first in, first out)
to maintain the exploration frontier.

### Algorithm

```
BFS(G, s):
  mark s as visited; distance[s] = 0; parent[s] = None
  Q ← queue containing just s
  while Q is not empty:
    u ← dequeue(Q)
    for each neighbour v of u:
      if v not visited:
        mark v as visited
        distance[v] = distance[u] + 1
        parent[v] = u
        enqueue(Q, v)
```

**Time complexity:** $O(|V| + |E|)$ — every vertex and edge is processed once.

### Correctness: BFS Finds Shortest Paths

**Theorem.** For any vertex $v$ reachable from $s$, $\text{BFS}$ sets
$\text{distance}[v] = \delta(s, v)$ — the true minimum hop count.

**Proof sketch.** BFS processes vertices in non-decreasing order of distance.
When vertex $v$ is first discovered from vertex $u$, all vertices at distance
$< \text{distance}[u] + 1$ have already been visited. If $v$ could be reached
in fewer hops, it would have been discovered earlier. $\square$

### Security Reading

BFS gives the **minimum hop attack path** — the fewest number of compromised
systems an attacker needs to traverse to reach the target. Each hop is a risk
event (a system compromise that could trigger alerts). The minimum-hop path
is the attacker's preferred route because it minimises detection opportunities.

From Module 00: *"Proof by contrapositive: if no path exists in the attack graph,
lateral movement is blocked."* BFS computes the constructive direction: if BFS
from $s$ does not reach $t$, then $\neg(s \rightsquigarrow t)$ — lateral movement
to $t$ is provably impossible from $s$.


## 3 · Depth-First Search (DFS)

**DFS** explores as deep as possible along each path before backtracking. It uses
a **stack** $S$ (LIFO — last in, first out), or equivalently, recursion.

### Algorithm

```
DFS(G, s):
  mark s as visited
  for each neighbour v of s:
    if v not visited:
      parent[v] = s
      DFS(G, v)     ← recursive call goes as deep as possible
```

**Time complexity:** $O(|V| + |E|)$ — same as BFS.

### Edge Classification

DFS classifies every edge it encounters into four types:

| Edge type | Definition | Security meaning |
|---|---|---|
| **Tree edge** | Leads to an undiscovered vertex | Primary traversal path |
| **Back edge** | Leads to an ancestor in the DFS tree | **Cycle detected** — looping behaviour |
| **Forward edge** | Leads to a descendant (directed graphs) | Shortcut in the attack graph |
| **Cross edge** | Leads to a non-ancestor, non-descendant | Cross-component connection |

### Cycle Detection via Back Edges

**Theorem.** A directed graph $G$ contains a cycle if and only if DFS discovers
a back edge — an arc $(u, v)$ where $v$ is an ancestor of $u$ in the DFS tree.

**Security reading.** A back edge in a network graph means a path from a "deeper"
internal system back to a "shallower" one — the structure needed for an attacker
to loop between systems, escalating privileges on each pass, or establishing
a persistent beachhead that continually re-compromises cleaned systems.

### BFS vs DFS: When to Use Which

| Question | Use |
|---|---|
| What is the minimum hop count to the target? | **BFS** |
| Is the target reachable at all? | Either (BFS is conventional) |
| Does the graph contain a cycle? | **DFS** (back edge detection) |
| What is the full reachability set? | Either |
| Find all paths (for attack graph enumeration)? | **DFS** |


## 4 · Formal Reachability as a Security Property

Connecting traversal back to the predicate logic of Module 02:

**Security guarantee (isolation holds):**

$$\forall s \in \text{EntryPoints},\; \forall t \in \text{Targets},\; t \notin R(s)$$

*"For every entry point, no target is reachable."*

**Violation condition** (negation via quantifier duality):

$$\exists s \in \text{EntryPoints},\; \exists t \in \text{Targets},\; t \in R(s)$$

*"There exists an entry point from which some target is reachable."*

**Computing the violation:** BFS from each entry point; union the reachability
sets; intersect with the target set:

$$\text{exposed targets} = \left(\bigcup_{s \in \text{EntryPoints}} R(s)\right) \cap \text{Targets}$$

If this set is non-empty, the isolation guarantee is violated. The exposed
targets are the witnesses to the $\exists$ violation.

**The contrapositive** (Module 00, proof by contrapositive):

$$t \notin R(s) \;\Rightarrow\; s \text{ cannot reach } t$$

If BFS from $s$ terminates without visiting $t$, then no directed path from
$s$ to $t$ exists — lateral movement from $s$ to $t$ is provably blocked.
This is the formal security guarantee that network segmentation provides.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AG\]

| Traversal concept | Security meaning |
|---|---|
| **BFS from entry point** | Compute full blast radius of initial compromise |
| **BFS distance $d(s,t)$** | Minimum hops for lateral movement — attacker's preferred path length |
| **$t \notin R(s)$** | Isolation guarantee — lateral movement provably impossible |
| **DFS back edge** | Cycle in attack graph — persistent re-compromise loop |
| **BFS layer** | "Hop distance from attacker" — each layer is one detection opportunity |
| **$\bigcup R(s) \cap \text{Targets}$** | Set of targets exposed across all entry points |
| **Queue (BFS) vs Stack (DFS)** | Layer-by-layer vs depth-first — different exploration orders, same reachability |

**The adversarial asymmetry, formalised.** The defender must ensure
$\forall s, t:\; t \notin R(s)$ — a universal guarantee. The attacker needs only
$\exists s, t:\; t \in R(s)$ — one reachable path. BFS from every entry point
is the exhaustive check of the defender's guarantee. Finding one path is all the
attacker needs.

---


## Python: Visualization & Verification

**1 · BFS Step-by-Step** — implement BFS from scratch, logging every step of the
queue, and animate the layer-by-layer expansion on an enterprise network graph.

**2 · DFS and Cycle Detection** — implement DFS with edge classification; detect
back edges; demonstrate cycle detection on a network containing a looping path.

**3 · Formal Reachability Audit** — compute the full reachability set from every
entry point; intersect with the target set; produce a formal audit report of
exposed targets, exactly mirroring the Module 02 audit framework.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import deque

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · BFS Step-by-Step on an Enterprise Network

We implement BFS from scratch — logging every step so the queue, visited set,
and distance assignments are fully transparent. We then visualise the result
as a layered diagram where each BFS layer corresponds to one hop of lateral movement.

The entry point is the internet-facing `web_server`. The question: how many hops
does it take to reach each internal system?


In [ ]:
# ── 1 · BFS Step-by-Step ─────────────────────────────────────────────────────

# Enterprise network as a directed graph
# Edge = "can reach / exploit connection to"
G = nx.DiGraph()

nodes = {
    'internet':          'external',
    'web_server':        'dmz',
    'api_server':        'dmz',
    'load_balancer':     'dmz',
    'db_primary':        'internal',
    'db_replica':        'internal',
    'admin_workstation': 'internal',
    'dev_laptop':        'internal',
    'log_server':        'internal',
    'backup_server':     'internal',
    'domain_controller': 'critical',
}
G.add_nodes_from(nodes.keys())

# Directed edges (attacker movement direction)
edges = [
    ('internet',     'web_server'),
    ('internet',     'api_server'),
    ('web_server',   'load_balancer'),
    ('web_server',   'api_server'),
    ('api_server',   'db_primary'),
    ('api_server',   'log_server'),
    ('load_balancer','db_primary'),
    ('db_primary',   'db_replica'),
    ('db_primary',   'backup_server'),
    ('db_primary',   'admin_workstation'),
    ('api_server',   'dev_laptop'),
    ('dev_laptop',   'admin_workstation'),
    ('admin_workstation', 'domain_controller'),
    ('admin_workstation', 'log_server'),
    ('log_server',   'backup_server'),
]
G.add_edges_from(edges)

# ── BFS from scratch ──────────────────────────────────────────────────────────
def bfs(G, source):
    visited  = {source}
    distance = {source: 0}
    parent   = {source: None}
    queue    = deque([source])
    layers   = {0: [source]}
    log      = []

    while queue:
        u = queue.popleft()
        for v in G.successors(u):
            if v not in visited:
                visited.add(v)
                distance[v] = distance[u] + 1
                parent[v]   = u
                queue.append(v)
                layers.setdefault(distance[v], []).append(v)
                log.append((u, v, distance[v]))

    return distance, parent, layers, log

source = 'web_server'
dist, parent, layers, log = bfs(G, source)

print(f"BFS from '{source}' — lateral movement distances:")
print(f"{'Vertex':<22} {'Hops':>5}  {'Via (parent)'}")
print("─" * 55)
for v in sorted(dist, key=lambda x: (dist[x], x)):
    p = parent[v] if parent[v] else '(entry point)'
    flag = '  ← TARGET' if nodes[v] == 'critical' else ''
    print(f"  {v:<20}  {dist[v]:>4}  {p}{flag}")

unreachable = [v for v in G.nodes() if v not in dist]
print(f"\nUnreachable from '{source}': {unreachable if unreachable else 'none — full reachability'}")

# ── Visualise BFS layers ───────────────────────────────────────────────────────
zone_colors = {'external': TS_GRAY, 'dmz': TS_AMBER,
               'internal': TS_BLUE, 'critical': TS_RED}
hop_colors  = [TS_GREEN, TS_AMBER, TS_BLUE, TS_RED, '#8e44ad']

fig, ax = plt.subplots(figsize=(13, 6))
max_layer = max(layers.keys())

pos = {}
for layer, verts in layers.items():
    for i, v in enumerate(sorted(verts)):
        pos[v] = (layer * 3.2, -(i - len(verts)/2) * 1.4)

node_colors = [hop_colors[min(dist.get(v, max_layer), len(hop_colors)-1)]
               for v in G.nodes()]

nx.draw_networkx_edges(G, pos, ax=ax, edge_color=TS_GRAY, alpha=0.4,
                       arrows=True, arrowsize=16, width=1.8,
                       connectionstyle='arc3,rad=0.08')
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=900, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7,
                        font_color='white', font_weight='bold')

# Highlight the path to domain_controller
if 'domain_controller' in dist:
    path_nodes = []
    v = 'domain_controller'
    while v is not None:
        path_nodes.append(v)
        v = parent[v]
    path_edges = [(path_nodes[i+1], path_nodes[i])
                  for i in range(len(path_nodes)-1)]
    nx.draw_networkx_edges(G, pos, edgelist=path_edges, ax=ax,
                           edge_color=TS_RED, width=3.5, alpha=0.9,
                           arrows=True, arrowsize=22,
                           connectionstyle='arc3,rad=0.08')

for lv in layers:
    ax.text(lv * 3.2, max(-(i - len(layers[lv])/2)*1.4
                          for i in range(len(layers[lv]))) + 1.0,
            f'Hop {lv}', ha='center', fontsize=9, color=TS_GRAY, style='italic')

ax.set_title(f"BFS from '{source}' — Lateral Movement Distances\n"
             f"Red path = shortest route to domain_controller ({dist.get('domain_controller','∞')} hops)",
             pad=12, fontsize=11)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The BFS table shows every system ordered by hop distance from `web_server`.
The `domain_controller` — the crown jewel of most enterprise networks, giving
the attacker control of all user accounts and group policies — is **3 hops away**.
The shortest path is highlighted in red: `web_server → api_server → db_primary
→ admin_workstation → domain_controller`.

Each hop is one more detection opportunity — and one more system that must be
compromised without triggering an alert. A defender who can detect anomalous
lateral movement at any single hop on the minimum path blocks the entire attack chain.

The layered layout makes the BFS structure visually clear: everything at the same
horizontal position was discovered in the same BFS wave — they are all the same
number of hops from the attacker's initial foothold.


### 2 · DFS and Back Edge Detection

We implement DFS with full edge classification. The key output is back edge
detection — back edges signal cycles, which in a network graph mean the attacker
can loop between systems for persistent access.


In [ ]:
# ── 2 · DFS with Edge Classification ────────────────────────────────────────

def dfs_classify(G, source):
    """
    DFS with edge classification.
    Returns (tree_edges, back_edges, forward_edges, cross_edges, finish_order)
    """
    WHITE, GRAY, BLACK = 0, 1, 2   # undiscovered, in-progress, finished
    color   = {v: WHITE for v in G.nodes()}
    parent  = {v: None  for v in G.nodes()}
    disc    = {}   # discovery time
    fin     = {}   # finish time
    time    = [0]

    tree_edges, back_edges, fwd_edges, cross_edges = [], [], [], []
    finish_order = []

    def dfs_visit(u):
        color[u]  = GRAY
        time[0]  += 1
        disc[u]   = time[0]

        for v in G.successors(u):
            if color[v] == WHITE:
                parent[v] = u
                tree_edges.append((u, v))
                dfs_visit(v)
            elif color[v] == GRAY:
                back_edges.append((u, v))      # back edge → cycle!
            elif color[v] == BLACK:
                if disc.get(u, 0) < disc.get(v, 0):
                    fwd_edges.append((u, v))   # forward edge
                else:
                    cross_edges.append((u, v)) # cross edge

        color[u] = BLACK
        time[0] += 1
        fin[u]   = time[0]
        finish_order.append(u)

    # Visit from source first, then any unvisited nodes
    dfs_visit(source)
    for v in G.nodes():
        if color[v] == WHITE:
            dfs_visit(v)

    return tree_edges, back_edges, fwd_edges, cross_edges, finish_order

# ── Run on base graph ─────────────────────────────────────────────────────────
te, be, fe, ce, fo = dfs_classify(G, 'web_server')

print(f"DFS Edge Classification from 'web_server':")
print(f"  Tree edges     ({len(te)}): lateral movement paths discovered")
print(f"  Back edges     ({len(be)}): CYCLES DETECTED" if be else
      f"  Back edges     ({len(be)}): none — acyclic")
print(f"  Forward edges  ({len(fe)})")
print(f"  Cross edges    ({len(ce)})")

# ── Plant a cycle: log_server can re-attack web_server ───────────────────────
G_cyclic = G.copy()
G_cyclic.add_edge('log_server', 'web_server')   # persistence loop
G_cyclic.add_edge('backup_server', 'api_server')

te_c, be_c, fe_c, ce_c, fo_c = dfs_classify(G_cyclic, 'web_server')

print(f"\nWith persistence loops added (log_server→web_server, backup_server→api_server):")
print(f"  Back edges ({len(be_c)}) — cycles:")
for u, v in be_c:
    print(f"    {u} → {v}  (back edge → cycle through {v})")

# ── Visualise DFS tree + back edges ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
pos_c = nx.spring_layout(G_cyclic, seed=42, k=2.8)
nc    = [TS_BLUE if nodes.get(v,'') != 'critical' else TS_RED
         for v in G_cyclic.nodes()]

tree_only = [(u,v) for u,v in G_cyclic.edges() if (u,v) in te_c]
back_only = [(u,v) for u,v in G_cyclic.edges() if (u,v) in be_c]
other     = [(u,v) for u,v in G_cyclic.edges()
             if (u,v) not in te_c and (u,v) not in be_c]

nx.draw_networkx_edges(G_cyclic, pos_c, ax=ax, edgelist=other,
                       edge_color=TS_GRAY, alpha=0.25, arrows=True,
                       arrowsize=14, width=1.2,
                       connectionstyle='arc3,rad=0.05')
nx.draw_networkx_edges(G_cyclic, pos_c, ax=ax, edgelist=tree_only,
                       edge_color=TS_BLUE, alpha=0.7, arrows=True,
                       arrowsize=18, width=2.0,
                       connectionstyle='arc3,rad=0.05')
nx.draw_networkx_edges(G_cyclic, pos_c, ax=ax, edgelist=back_only,
                       edge_color=TS_RED, alpha=0.95, arrows=True,
                       arrowsize=22, width=3.0, style='dashed',
                       connectionstyle='arc3,rad=0.3')
nx.draw_networkx_nodes(G_cyclic, pos_c, ax=ax, node_color=nc,
                       node_size=900, alpha=0.92)
nx.draw_networkx_labels(G_cyclic, pos_c, ax=ax, font_size=7,
                        font_color='white', font_weight='bold')

legend = [
    mpatches.Patch(color=TS_BLUE,  label='Tree edge (DFS path)'),
    mpatches.Patch(color=TS_RED,   label='Back edge (CYCLE — persistence loop)'),
    mpatches.Patch(color=TS_GRAY,  label='Other edges'),
]
ax.legend(handles=legend, loc='upper left', fontsize=9)
ax.set_title('DFS Edge Classification — Back Edges Reveal Persistence Loops',
             pad=12, fontsize=11)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The original graph has no back edges — it is a DAG. The attacker can move forward
through the network but cannot loop back. A defender who can isolate the
`admin_workstation` after detecting an intrusion breaks all paths to the domain
controller cleanly.

After adding the persistence loops, two back edges appear (red dashed arcs). The
`log_server → web_server` edge creates a cycle: an attacker who compromises the
log server can use it to re-attack the web server even if the web server is patched.
This is the formal structure of a **persistent threat** — the attacker maintains
a foothold that continuously regenerates access.

DFS back edge detection is not just a theoretical exercise. Real-world attack
graph analysis tools (BloodHound, for example) use exactly this to identify
cycles in Active Directory permission graphs — the back edges in the AD graph
are the "golden ticket" attack paths.


### 3 · Formal Reachability Audit

We run the full Module 02-style formal audit on the network graph. For every
defined entry point, we compute the reachability set using BFS, then intersect
with the set of designated high-value targets. The output is a structured audit
report — findings expressed as witnesses to the $\exists$ violation condition.


In [ ]:
# ── 3 · Formal Reachability Audit ────────────────────────────────────────────

entry_points = {'internet', 'dev_laptop'}
targets      = {'domain_controller', 'db_primary', 'backup_server'}

def reachability_set(G, source):
    """BFS reachability — returns set of all vertices reachable from source."""
    dist, _, _, _ = bfs(G, source)
    return set(dist.keys()) - {source}

# Compute reachability from every entry point
reach = {s: reachability_set(G, s) for s in entry_points}

print("=" * 65)
print("FORMAL REACHABILITY AUDIT")
print("=" * 65)
print(f"\nSecurity property:")
print(f"  ∀s ∈ EntryPoints, ∀t ∈ Targets, t ∉ R(s)")
print(f"  (No target is reachable from any entry point)")
print(f"\nViolation condition:")
print(f"  ∃s ∈ EntryPoints, ∃t ∈ Targets, t ∈ R(s)")
print()

findings = []
for s in sorted(entry_points):
    exposed = reach[s] & targets
    print(f"Entry point: {s}")
    print(f"  Reachability set R({s}): {sorted(reach[s])}")
    print(f"  Targets in R({s}):       {sorted(exposed)}")
    for t in sorted(exposed):
        dist_s, parent_s, _, _ = bfs(G, s)
        # Reconstruct shortest path
        path, v = [], t
        while v is not None:
            path.append(v); v = parent_s[v]
        path.reverse()
        findings.append((s, t, dist_s[t], path))
        print(f"    ⚠  FINDING: {s} ⇝ {t}  ({dist_s[t]} hops): {' → '.join(path)}")
    print()

print("=" * 65)
print(f"Summary: {len(findings)} violation(s) found")
if findings:
    print(f"  Property FALSE — isolation guarantee violated")
    print(f"  Witnesses:")
    for s, t, d, path in findings:
        print(f"    ({s}, {t}): path of {d} hops")
else:
    print(f"  Property TRUE — no targets reachable from any entry point ✓")
print("=" * 65)

# ── Visualise exposed targets ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
pos_g = nx.spring_layout(G, seed=99, k=2.5)

exposed_targets = {t for _,t,_,_ in findings}
def node_color(v):
    if v in entry_points:    return TS_GREEN
    if v in exposed_targets: return TS_RED
    if v in targets:         return TS_AMBER
    return TS_BLUE

nc = [node_color(v) for v in G.nodes()]
nx.draw_networkx_edges(G, pos_g, ax=ax, edge_color=TS_GRAY,
                       alpha=0.35, arrows=True, arrowsize=14, width=1.5,
                       connectionstyle='arc3,rad=0.06')

# Draw all shortest attack paths in amber
for s, t, d, path in findings:
    path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    nx.draw_networkx_edges(G, pos_g, edgelist=path_edges, ax=ax,
                           edge_color=TS_AMBER, width=3, alpha=0.85,
                           arrows=True, arrowsize=20,
                           connectionstyle='arc3,rad=0.06')

nx.draw_networkx_nodes(G, pos_g, ax=ax, node_color=nc, node_size=900, alpha=0.92)
nx.draw_networkx_labels(G, pos_g, ax=ax, font_size=7,
                        font_color='white', font_weight='bold')

legend = [
    mpatches.Patch(color=TS_GREEN, label='Entry point'),
    mpatches.Patch(color=TS_RED,   label='Exposed target (violation witness)'),
    mpatches.Patch(color=TS_AMBER, label='Unreached target / attack path'),
    mpatches.Patch(color=TS_BLUE,  label='Other vertex'),
]
ax.legend(handles=legend, loc='upper left', fontsize=9)
ax.set_title('Formal Reachability Audit — Exposed Targets and Attack Paths',
             pad=12, fontsize=11)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The audit report mirrors the Module 02 `SecurityProperty` framework exactly —
formal property, violation condition, and specific witnesses with full evidence
(the shortest path from entry to target).

Every defined target is reachable from the internet entry point. The amber paths
show the minimum-hop routes. The domain controller — the highest-value target —
is reachable in 4 hops from internet, 3 from `dev_laptop`.

**The formal security statement is FALSE** — the isolation guarantee is violated.
The witnesses are the (entry point, target, path) triples. A security team
seeing this output knows exactly which network segments need to be added to
break those paths. That is the subject of Unit 03.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. BIDIRECTIONAL BFS
#    Standard BFS from the source finds all vertices reachable from the source.
#    To find the shortest path between TWO specific vertices s and t efficiently,
#    bidirectional BFS runs BFS from both ends simultaneously and stops when
#    the frontiers meet.
#    Implement bidirectional_bfs(G, s, t) → (distance, path).
#    Verify it gives the same result as standard BFS for the web_server → domain_controller path.
#
# 2. STRONGLY CONNECTED COMPONENTS AS BLAST ZONES
#    nx.strongly_connected_components(G) computes SCCs.
#    Apply it to G_cyclic (the graph with persistence loops).
#    For each SCC with more than one node: if the attacker compromises any member,
#    every other member is reachable. Print a "blast zone report" listing each SCC,
#    its members, and which high-value targets it contains.
#
# 3. MULTI-SOURCE BFS
#    Suppose the attacker has compromised ALL entry points simultaneously
#    (a coordinated attack). Implement multi_source_bfs(G, sources) that
#    starts BFS from all sources at once (add them all to the initial queue
#    with distance 0). Compare the reachability and minimum distances vs.
#    single-source BFS. How does a coordinated attack change the blast radius?

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **BFS** | Queue-based layer-by-layer traversal | Minimum hop count; full blast radius |
| **DFS** | Stack-based depth-first traversal | Cycle detection; all path enumeration |
| **$R(s)$** | Reachability set of $s$ | Every system at risk if $s$ is compromised |
| **$d(s,t)$** | Minimum hop distance | Fewest compromises attacker needs |
| **Back edge** | Arc to a DFS ancestor | Cycle — persistence loop |
| **$t \notin R(s)$** | Isolation guarantee | Contrapositive: lateral movement provably blocked |
| **$\exists s,t: t \in R(s)$** | Violation witness | A reachable target — a finding |

---

## Up Next

**Module 05 · Unit 02 — Shortest Path and Attack Graphs**

BFS finds the minimum-hop path. But not all hops are equal — some exploits
are easy (CVSS 9.8) and some are hard (CVSS 3.1). Dijkstra's algorithm finds
the minimum-**cost** path on a weighted graph, where edge weights represent
exploit difficulty. The attacker follows the cheapest path, not the shortest
one in hops.

$\rightarrow$ `modules/module-05/unit-02-shortest-path-attack-graphs.ipynb`
